# Dataset EDA

In [31]:
# Imports

import pandas as pd
from sklearn.preprocessing import OneHotEncoder

In [32]:
# DataLoad (CSV)

customers = pd.read_csv(r"C:\Users\sailj\OneDrive\文档\GitHub\SmartCommerce Analytics\dataset\customers.csv")
print("customer.csv")
print()
print(customers.head())

customer.csv

  customer_id signup_date  age             gender country         segment  \
0      C00000  2020-02-13   28                  M      BR         Premium   
1      C00001  2021-10-01   22  Prefer not to say      FR         Regular   
2      C00002  2022-06-26   30                  F      US             VIP   
3      C00003  2021-12-21   48                  F      US         Premium   
4      C00004  2021-09-16   37                  M      CA  Budget Shopper   

   is_churned  lifetime_value  email_opt_in  has_app  
0           0         1595.27             0        0  
1           0         1160.61             1        0  
2           0         3093.32             1        1  
3           1         2131.08             1        0  
4           0          583.39             0        1  


In [33]:
transactions = pd.read_csv(r"C:\Users\sailj\OneDrive\文档\GitHub\SmartCommerce Analytics\dataset\transactions.csv")
print("transaction.csv")
print()
print(transactions.head())

transaction.csv

  transaction_id customer_id product_id     transaction_date  quantity  \
0        T077767      C08119      P0932  2023-01-01 00:23:39         1   
1        T021376      C08883      P0889  2023-01-01 00:26:51         1   
2        T117783      C05855      P0065  2023-01-01 00:45:53         1   
3        T035185      C09437      P0200  2023-01-01 01:04:28         1   
4        T092643      C00760      P0881  2023-01-01 01:06:45         1   

   unit_price  total_amount  discount_applied     status payment_method  \
0       23.87         23.87                 0   refunded         paypal   
1       31.53         31.53                50   refunded    credit_card   
2       67.75         67.75                 0  completed      apple_pay   
3      151.06        151.06                 5  cancelled      apple_pay   
4       11.97         11.97                50  completed         paypal   

   shipping_cost  
0           8.15  
1           6.13  
2           0.00  
3          

In [34]:
products = pd.read_csv(r"C:\Users\sailj\OneDrive\文档\GitHub\SmartCommerce Analytics\dataset\products.csv")
print("products.csv")
print()
print(products.head())

products.csv

  product_id                  product_name         category      brand  \
0      P0000             PetLife Health #0           Health    PetLife   
1      P0001              GlowUp Beauty #1           Beauty     GlowUp   
2      P0002  EcoLiving Office Supplies #2  Office Supplies  EcoLiving   
3      P0003      DigiTools Electronics #3      Electronics  DigiTools   
4      P0004      StyleMax Pet Supplies #4     Pet Supplies   StyleMax   

    price  avg_rating  num_ratings  stock_quantity  discount_pct  is_featured  \
0   34.82         4.0          137             126             0            0   
1   53.35         5.0          494              51            10            0   
2   51.45         3.5          433             928             0            0   
3  179.90         4.0          267             127             0            1   
4   12.54         3.7          402             580             0            0   

   weight_kg  
0       0.35  
1       1.76  
2       0

In [35]:
# Merge
df = pd.merge(transactions , customers , on="customer_id")
df = pd.merge(df , products , on="product_id")

# Removing unnecessary columns
df = df.drop(["product_id" , "transaction_date" , "signup_date" , "product_name" , "stock_quantity" , "discount_pct" ,  "weight_kg" , "category" , "brand"] , axis=1)

print(df.head())

  transaction_id customer_id  quantity  unit_price  total_amount  \
0        T077767      C08119         1       23.87         23.87   
1        T021376      C08883         1       31.53         31.53   
2        T117783      C05855         1       67.75         67.75   
3        T035185      C09437         1      151.06        151.06   
4        T092643      C00760         1       11.97         11.97   

   discount_applied     status payment_method  shipping_cost  age  ...  \
0                 0   refunded         paypal           8.15   43  ...   
1                50   refunded    credit_card           6.13   25  ...   
2                 0  completed      apple_pay           0.00   31  ...   
3                 5  cancelled      apple_pay           0.00   27  ...   
4                50  completed         paypal           5.31   50  ...   

  country         segment is_churned  lifetime_value  email_opt_in  has_app  \
0      US         Premium          0         3118.98             0 

In [36]:
# OHE FOR paymnet_method

ohe = OneHotEncoder(handle_unknown="ignore" , sparse_output=False)

array_encoded_payment_method = ohe.fit_transform(df[["payment_method"]])
encoded_payment_method = pd.DataFrame(array_encoded_payment_method , columns=ohe.get_feature_names_out(["payment_method"]))
df = df.drop(["payment_method"] , axis=1)
df = pd.concat([df , encoded_payment_method] , axis=1)

In [37]:
# OHE FOR status

array_encoded_status = ohe.fit_transform(df[["status"]])
encoded_status = pd.DataFrame(array_encoded_status , columns=ohe.get_feature_names_out(["status"]))
df = df.drop(["status"] , axis=1)
df = pd.concat([df , encoded_status] , axis=1)

In [38]:
# Group By ID

df = df.groupby("customer_id").agg({
        "quantity" : "sum" ,
        "unit_price" : "mean" ,
        "total_amount" : "sum" ,    
        "discount_applied" : ["sum" , "mean"] ,
        "shipping_cost" : ["sum" , "mean"] ,
        "age" : "first" ,
        "lifetime_value" : "max" ,
        "transaction_id" : "count" ,
        "email_opt_in" : "max",
        "has_app" : "max",
        "price" : "mean" ,
        "avg_rating" : "mean" ,
        "num_ratings" : "mean" , 
        "gender" : "first" ,
        "country" : "first" ,
        "segment" : "first" ,
        "payment_method_apple_pay" : "sum" ,
        "payment_method_bank_transfer" : "sum" , 
        "payment_method_credit_card" : "sum" , 
        "payment_method_debit_card" : "sum" , 
        "payment_method_google_pay" : "sum" , 
        "payment_method_paypal" : "sum" ,
        "status_cancelled" : "sum" , 
        "status_completed" : "sum" , 
        "status_pending" : "sum" , 
        "status_refunded" : "sum",
        "is_churned" : "first"
})

# Making defalut index
df = df.reset_index()

# Filterring column name
df.columns = ['_'.join(col).strip('_') for col in df.columns]
df = df.drop(["customer_id"] , axis=1)

print(df.head())

   quantity_sum  unit_price_mean  total_amount_sum  discount_applied_sum  \
0            32        34.675200            970.60                   115   
1            22        32.725833            786.16                   100   
2            39        58.933448           2112.49                   195   
3            40        31.813462           1341.06                   210   
4             4        46.135000            184.54                    25   

   discount_applied_mean  shipping_cost_sum  shipping_cost_mean  age_first  \
0               4.600000             144.08            5.763200         28   
1               8.333333              42.33            3.527500         22   
2               6.724138             131.13            4.521724         30   
3               8.076923             118.08            4.541538         48   
4               6.250000              19.31            4.827500         37   

   lifetime_value_max  transaction_id_count  ...  \
0             1595.27 